# Studio 2 — Training Final di Google Colab GPU

**Kelompok:** BAI  
**Repo:** https://github.com/Aryasatyaa45/Studio2_BAI  
**Deadline Final:** 25 Oktober 2026, 23:59 WIB

## Panduan Singkat
1. Upload folder `train/` ke Google Drive (letakkan di `MyDrive/Studio2/train/`)
2. Jalankan cell satu per satu dari atas ke bawah
3. Pastikan runtime GPU: **Runtime → Change runtime type → GPU**
4. Model terbaik otomatis tersimpan di `submission/model.pth`
5. Push ke GitHub di cell terakhir

## 1. Mount Google Drive & Clone Repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repository kelompok
!git clone https://github.com/Aryasatyaa45/Studio2_BAI.git
%cd Studio2_BAI
!ls -la

## 2. Verifikasi GPU

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('GPU memory:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')
else:
    print('WARNING: GPU tidak terdeteksi! Ganti runtime ke GPU: Runtime → Change runtime type → GPU')

## 3. Siapkan Dataset

Pastikan folder `train/` sudah di-upload ke Google Drive di path: `MyDrive/Studio2/train/`

In [ ]:
import shutil
from pathlib import Path

# Path dataset di Google Drive
drive_data = Path('/content/drive/MyDrive/Studio2/train')
local_data = Path('/content/Studio2_BAI/dataset_train')

if not drive_data.exists():
    raise FileNotFoundError(f"Dataset tidak ditemukan di {drive_data}. Upload folder train/ ke Google Drive terlebih dahulu.")

# Copy dataset ke runtime Colab (lebih cepat daripada baca langsung dari Drive)
if local_data.exists():
    shutil.rmtree(local_data)
shutil.copytree(drive_data, local_data)
print(f'Dataset disalin ke {local_data}')

# Verifikasi jumlah gambar per kelas
for cls_dir in sorted(local_data.iterdir()):
    if cls_dir.is_dir():
        n = len(list(cls_dir.glob('*.jpg')))
        print(f'{cls_dir.name}: {n} gambar')

## 4. Install Dependencies

In [ ]:
!pip install -q torch torchvision torchinfo pillow numpy

---

## 5. EKSPERIMEN 1 — Baseline ResNet-18 (8 epoch)

Ini adalah baseline yang sama dengan Preview 1. Jalankan untuk verifikasi hasil di GPU.

In [ ]:
!python train.py --data /content/Studio2_BAI/dataset_train --epochs 8 --batch-size 64 --lr 3e-4

In [ ]:
# Backup model baseline
import shutil
shutil.copy('submission/model.pth', 'submission/model_baseline.pth')
print('Model baseline dibackup ke submission/model_baseline.pth')

---

## 6. EKSPERIMEN 2 — ResNet-18 + Weighted Loss + Augmentasi Agresif (15 epoch)

**Strategi:**
- Class weights untuk menangani imbalance (kertas_kardus paling sedikit)
- Augmentasi lebih agresif: rotation, grayscale, stronger color jitter
- Lebih banyak epoch (15) karena GPU cepat
- Early stopping patience 5 berdasarkan Macro-F1

In [ ]:
# Buat script training eksperimen 2
exp2_script = '''
import argparse
import random
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms

from submission.model import get_model, get_transform

CLASSES = [
    "botol_plastik",
    "gelas_plastik",
    "kertas_kardus",
    "saset_kemasan",
    "styrofoam",
]

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def make_local_split(source_dir, split_dir, validation_fraction, seed):
    train_dir = split_dir / "my_train"
    validation_dir = split_dir / "my_val"
    if train_dir.exists() and validation_dir.exists():
        return train_dir, validation_dir
    rng = random.Random(seed)
    for class_name in CLASSES:
        source_class = source_dir / class_name
        files = sorted(source_class.glob("*.jpg"))
        rng.shuffle(files)
        validation_count = int(len(files) * validation_fraction)
        for index, source_file in enumerate(files):
            target_root = validation_dir if index < validation_count else train_dir
            target_file = target_root / class_name / source_file.name
            target_file.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_file, target_file)
    return train_dir, validation_dir

def macro_f1(targets, predictions, num_classes):
    confusion = torch.zeros((num_classes, num_classes), dtype=torch.long)
    for target, prediction in zip(targets, predictions):
        confusion[target, prediction] += 1
    f1_values = []
    for class_index in range(num_classes):
        tp = confusion[class_index, class_index].item()
        fp = confusion[:, class_index].sum().item() - tp
        fn = confusion[class_index, :].sum().item() - tp
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1_values.append(2 * precision * recall / (precision + recall) if precision + recall else 0.0)
    return sum(f1_values) / num_classes

def evaluate(model, loader, device):
    model.eval()
    targets, predictions = [], []
    with torch.inference_mode():
        for images, labels in loader:
            logits = model(images.to(device))
            batch_predictions = logits.argmax(dim=1).cpu().tolist()
            predictions.extend(batch_predictions)
            targets.extend(labels.tolist())
    accuracy = sum(t == p for t, p in zip(targets, predictions)) / len(targets)
    return macro_f1(targets, predictions, len(CLASSES)), accuracy

def load_pretrained_backbone(model):
    weights = models.ResNet18_Weights.IMAGENET1K_V1
    pretrained = models.resnet18(weights=weights).state_dict()
    pretrained = {k: v for k, v in pretrained.items() if not k.startswith("fc.")}
    model.load_state_dict(pretrained, strict=False)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", required=True, type=Path)
    parser.add_argument("--epochs", type=int, default=15)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--validation-fraction", type=float, default=0.15)
    parser.add_argument("--work-dir", type=Path, default=Path("local_data"))
    parser.add_argument("--patience", type=int, default=5)
    args = parser.parse_args()

    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_dir, validation_dir = make_local_split(
        args.data.resolve(), args.work_dir.resolve(), args.validation_fraction, args.seed
    )

    # Augmentasi agresif untuk training
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
        transforms.RandomGrayscale(p=0.1),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    validation_dataset = datasets.ImageFolder(validation_dir, transform=get_transform())
    train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)

    # WeightedRandomSampler untuk oversampling kelas minoritas
    targets = torch.tensor(train_dataset.targets)
    class_counts = torch.bincount(targets, minlength=len(CLASSES)).float()
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[targets]
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

    loader_kwargs = {"batch_size": args.batch_size, "num_workers": 2, "pin_memory": True}
    train_loader = DataLoader(train_dataset, sampler=sampler, **loader_kwargs)
    validation_loader = DataLoader(validation_dataset, shuffle=False, **loader_kwargs)

    model = get_model()
    load_pretrained_backbone(model)
    model.to(device)

    # Weighted loss
    loss_weight = (class_counts.sum() / (len(CLASSES) * class_counts)).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=loss_weight, label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, args.epochs)

    submission_dir = Path("submission")
    submission_dir.mkdir(exist_ok=True)
    best_macro_f1 = -1.0
    patience_counter = 0
    print(f"device={device}  train={len(train_dataset)}  val={len(validation_dataset)}")

    for epoch in range(1, args.epochs + 1):
        model.train()
        total_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
        scheduler.step()

        current_macro_f1, current_accuracy = evaluate(model, validation_loader, device)
        average_loss = total_loss / len(train_dataset)
        print(f"epoch={epoch:02d} loss={average_loss:.4f} val_macro_f1={current_macro_f1:.4f} val_accuracy={current_accuracy:.4f}")

        if current_macro_f1 > best_macro_f1:
            best_macro_f1 = current_macro_f1
            torch.save(model.state_dict(), submission_dir / "model.pth")
            print(f"saved=submission/model.pth best_macro_f1={best_macro_f1:.4f}")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= args.patience:
                print(f"Early stopping at epoch {epoch}")
                break

    print(f"\nTraining selesai. Best Macro-F1: {best_macro_f1:.4f}")

if __name__ == "__main__":
    main()
'''

with open('train_exp2_aggressive.py', 'w') as f:
    f.write(exp2_script)
print('Script train_exp2_aggressive.py dibuat')

In [ ]:
# Jalankan eksperimen 2
!python train_exp2_aggressive.py --data /content/Studio2_BAI/dataset_train --epochs 15 --batch-size 64 --lr 3e-4 --patience 5

In [ ]:
# Backup model eksperimen 2
import shutil
shutil.copy('submission/model.pth', 'submission/model_exp2.pth')
print('Model exp2 dibackup')

---

## 7. EKSPERIMEN 3 — ResNet-34 Pretrained (12 epoch)

ResNet-34 lebih dalam dari ResNet-18, berpotensi menangkap fitur yang lebih baik.
**Catatan:** Submission model.py harus diubah ke ResNet-34 jika model ini terpilih!

In [ ]:
# Buat model.py untuk ResNet-34 (simpan sebagai model_resnet34.py)
model34_code = '''
import torch.nn as nn
from torchvision import models, transforms

NUM_CLASSES = 5

def get_model():
    model = models.resnet34(weights=None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model

def get_transform():
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
'''

with open('submission/model_resnet34.py', 'w') as f:
    f.write(model34_code)

# Buat script training ResNet-34
exp3_script = '''
import argparse
import random
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms

CLASSES = ["botol_plastik", "gelas_plastik", "kertas_kardus", "saset_kemasan", "styrofoam"]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def make_local_split(source_dir, split_dir, val_frac, seed):
    train_dir = split_dir / "my_train"
    val_dir = split_dir / "my_val"
    if train_dir.exists() and val_dir.exists():
        return train_dir, val_dir
    rng = random.Random(seed)
    for class_name in CLASSES:
        files = sorted((source_dir / class_name).glob("*.jpg"))
        rng.shuffle(files)
        val_count = int(len(files) * val_frac)
        for i, sf in enumerate(files):
            tr = val_dir if i < val_count else train_dir
            tf = tr / class_name / sf.name
            tf.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(sf, tf)
    return train_dir, val_dir

def macro_f1(targets, predictions, num_classes):
    confusion = torch.zeros((num_classes, num_classes), dtype=torch.long)
    for t, p in zip(targets, predictions):
        confusion[t, p] += 1
    f1s = []
    for c in range(num_classes):
        tp = confusion[c, c].item()
        fp = confusion[:, c].sum().item() - tp
        fn = confusion[c, :].sum().item() - tp
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    return sum(f1s) / num_classes

def evaluate(model, loader, device):
    model.eval()
    targets, predictions = [], []
    with torch.inference_mode():
        for images, labels in loader:
            preds = model(images.to(device)).argmax(dim=1).cpu().tolist()
            predictions.extend(preds)
            targets.extend(labels.tolist())
    acc = sum(t == p for t, p in zip(targets, predictions)) / len(targets)
    return macro_f1(targets, predictions, len(CLASSES)), acc

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", required=True, type=Path)
    parser.add_argument("--epochs", type=int, default=12)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--validation-fraction", type=float, default=0.15)
    parser.add_argument("--work-dir", type=Path, default=Path("local_data"))
    parser.add_argument("--patience", type=int, default=5)
    args = parser.parse_args()

    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_dir, val_dir = make_local_split(args.data.resolve(), args.work_dir.resolve(), args.validation_fraction, args.seed)

    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
        transforms.RandomGrayscale(p=0.1),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

    train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
    val_dataset = datasets.ImageFolder(val_dir, transform=transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]))

    targets = torch.tensor(train_dataset.targets)
    class_counts = torch.bincount(targets, minlength=len(CLASSES)).float()
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[targets]
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

    kwargs = {"batch_size": args.batch_size, "num_workers": 2, "pin_memory": True}
    train_loader = DataLoader(train_dataset, sampler=sampler, **kwargs)
    val_loader = DataLoader(val_dataset, shuffle=False, **kwargs)

    model = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
    model.to(device)

    loss_weight = (class_counts.sum() / (len(CLASSES) * class_counts)).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=loss_weight, label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, args.epochs)

    submission_dir = Path("submission")
    submission_dir.mkdir(exist_ok=True)
    best_f1 = -1.0
    patience_cnt = 0
    print(f"device={device} train={len(train_dataset)} val={len(val_dataset)}")

    for epoch in range(1, args.epochs + 1):
        model.train()
        total_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
        scheduler.step()

        f1, acc = evaluate(model, val_loader, device)
        avg_loss = total_loss / len(train_dataset)
        print(f"epoch={epoch:02d} loss={avg_loss:.4f} val_macro_f1={f1:.4f} val_accuracy={acc:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), submission_dir / "model.pth")
            print(f"saved=submission/model.pth best_macro_f1={best_f1:.4f}")
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= args.patience:
                print(f"Early stopping at epoch {epoch}")
                break

    print(f"\nTraining selesai. Best Macro-F1: {best_f1:.4f}")

if __name__ == "__main__":
    main()
'''

with open('train_exp3_resnet34.py', 'w') as f:
    f.write(exp3_script)
print('Script train_exp3_resnet34.py dibuat')

In [ ]:
# Jalankan eksperimen 3
!python train_exp3_resnet34.py --data /content/Studio2_BAI/dataset_train --epochs 12 --batch-size 64 --lr 3e-4 --patience 5

In [ ]:
# Backup model eksperimen 3
import shutil
shutil.copy('submission/model.pth', 'submission/model_exp3.pth')
print('Model exp3 dibackup')

---

## 8. Pilih Model Terbaik & Evaluasi

Bandingkan hasil ketiga eksperimen, lalu pilih model terbaik untuk submission Final.

In [ ]:
# Tabel perbandingan (isi manual setelah lihat hasil training di atas)
print("=== PERBANDINGAN EKSPERIMEN ===")
print("Eksperimen 1 (Baseline ResNet-18):    Macro-F1 = ???")
print("Eksperimen 2 (Weighted+Augmentasi):   Macro-F1 = ???")
print("Eksperimen 3 (ResNet-34):             Macro-F1 = ???")
print("\nPilih model dengan Macro-F1 tertinggi, lalu copy ke submission/model.pth")

In [ ]:
# CONTOH: Jika eksperimen 2 terbaik, uncomment baris berikut:
# import shutil
# shutil.copy('submission/model_exp2.pth', 'submission/model.pth')
# print('Model terbaik (exp2) disalin ke submission/model.pth')

# CONTOH: Jika eksperimen 3 (ResNet-34) terbaik, uncomment baris berikut:
# shutil.copy('submission/model_exp3.pth', 'submission/model.pth')
# shutil.copy('submission/model_resnet34.py', 'submission/model.py')  # WAJIB ganti model.py!
# print('Model terbaik (exp3 ResNet-34) disalin + model.py diperbarui')

In [ ]:
# Evaluasi final dengan evaluator resmi
!python evaluate.py --submission ./submission --data ./local_data/my_val --out ./final_metrics.json

In [ ]:
# Lihat hasil evaluasi
import json
with open('final_metrics.json') as f:
    metrics = json.load(f)
print(f"Macro-F1: {metrics['macro_f1']:.4f}")
print(f"Accuracy: {metrics['accuracy']:.4f}")
for cls, f1 in metrics['per_class_f1'].items():
    print(f"  F1 {cls:<15} {f1:.4f}")

---

## 9. Push ke GitHub (FINAL SUBMISSION)

**PENTING:** Jalankan cell ini SETELAH memilih model terbaik dan memverifikasi evaluator lolos!

In [ ]:
# Konfigurasi Git
!git config --global user.email "aryasatya@student.itera.ac.id"
!git config --global user.name "Aryasatya"

# Status
!git status

In [ ]:
# Add dan commit
!git add submission/model.pth submission/model.py README.md
!git commit -m "Final submission: model terbaik Studio 2"
!git log --oneline -3

In [ ]:
# Push ke GitHub
# Jika diminta username/password, gunakan Personal Access Token sebagai password
!git push origin main

---

## 10. Verifikasi Final

Setelah push, cek repository di browser: https://github.com/Aryasatyaa45/Studio2_BAI

Pastikan:
- [ ] `submission/model.py` ada
- [ ] `submission/model.pth` ada (ukuran ~40-80 MB)
- [ ] `evaluate.py` ada
- [ ] README.md berisi nama lengkap + NIM anggota
- [ ] Commit terakhir sebelum 25 Okt 2026, 23:59 WIB